In [ ]:
# Khởi tạo môi trường và Spark Session
!pip install pyspark delta-spark mlflow boto3 sentence-transformers lancedb requests python-dotenv pandas -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import os
import boto3
import requests
import time
from google.colab import userdata

# Khởi tạo Spark với cấp phát RAM hợp lý cho Colab Free
spark = SparkSession.builder \
    .appName("MovieLens25M-Pipeline") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

print("Spark Session Initialized!")

In [ ]:
# 📍 Bước 0 — Tải dữ liệu thô từ R2 (Nếu chưa có)
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('R2_ACCESS_KEY')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('R2_SECRET_KEY')
os.environ['AWS_ENDPOINT_URL'] = userdata.get('R2_ENDPOINT')
os.environ['TMDB_API_KEY'] = userdata.get('TMDB_API_KEY')

s3 = boto3.client('s3',
    endpoint_url=os.environ['AWS_ENDPOINT_URL'],
    aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
)

bucket = "movie-mlops"
files_to_download = ['movies.csv', 'links.csv', 'tags.csv', 'ratings.csv', 'genome-scores.csv', 'genome-tags.csv']

print("Đang tải 6 files từ R2 Data Lake...")
for f in files_to_download:
    if not os.path.exists(f'/content/{f}'):
        print(f"Downloading {f}...")
        s3.download_file(bucket, f"raw/{f}", f"/content/{f}")
print("Tải hoàn tất!")

In [ ]:
# 📍 Bước 1 — Lọc Quality Movies từ ratings.csv (PySpark)
# 🎯 Mục đích: Lọc bỏ phim rác (dưới 50 lượt đánh giá) và tính điểm trung bình

print("Đang xử lý ratings.csv (25M dòng) bằng PySpark...")
ratings_sdf = spark.read.csv("/content/ratings.csv", header=True, inferSchema=True)

quality_movies_sdf = ratings_sdf.groupBy("movieId") \
    .agg(
        F.round(F.avg("rating"), 2).alias("avg_rating"),
        F.count("rating").alias("rating_count")
    ) \
    .filter(F.col("rating_count") >= 50)

# Lưu vào Pandas để xử lý các bước tiếp theo (tránh giữ Spark DF quá lâu)
quality_movies_df = quality_movies_sdf.toPandas()
print(f"Số lượng Quality Movies: {len(quality_movies_df)}")

In [ ]:
# 📍 Bước 2 — Trích xuất Genome Tags (Pandas)
# 🎯 Mục đích: Lấy ra các tag chuẩn mức độ cao (>0.8) cho từng bộ phim

print("Đang xử lý Genome Tags...")
# Đọc file lớn bằng Pandas chunks nếu thiếu RAM, nhưng 15M dòng thường chiếm ~300MB RAM, Colab 12GB có thể kham được.
genome_scores = pd.read_csv('/content/genome-scores.csv')
genome_tags = pd.read_csv('/content/genome-tags.csv')

# Chỉ lấy tag có độ tương quan cao
high_rel_scores = genome_scores[genome_scores['relevance'] >= 0.8]

# Join lấy tên tag
genome_merged = high_rel_scores.merge(genome_tags, on='tagId', how='left')

# Sort relevance giảm dần và lấy top 10
genome_merged = genome_merged.sort_values(['movieId', 'relevance'], ascending=[True, False])
top_genome_tags = genome_merged.groupby('movieId').head(10)

# Gom thành 1 chuỗi
genome_tags_grouped = top_genome_tags.groupby('movieId')['tag'].apply(lambda x: ', '.join(x)).reset_index(name='genome_tags')
print(f"Đã trích xuất genome tags cho {len(genome_tags_grouped)} phim")

In [ ]:
# 📍 Bước 3 — Trích xuất User Tags (Pandas)
# 🎯 Mục đích: Lấy Top 5 user tags phổ biến nhất, bổ sung những từ khóa lóng của cộng đồng

print("Đang xử lý User Tags...")
tags_df = pd.read_csv('/content/tags.csv')
tags_df['tag'] = tags_df['tag'].astype(str).str.lower().str.strip()

tag_counts = tags_df.groupby(['movieId', 'tag']).size().reset_index(name='count')
tag_counts = tag_counts.sort_values(['movieId', 'count'], ascending=[True, False])
top_user_tags = tag_counts.groupby('movieId').head(5)

user_tags_grouped = top_user_tags.groupby('movieId')['tag'].apply(lambda x: ', '.join(x)).reset_index(name='user_tags')
print(f"Đã trích xuất user tags cho {len(user_tags_grouped)} phim")

In [ ]:
# 📍 Bước 4 — Merge tất cả (Pandas)
# 🎯 Mục đích: Gom tất cả dữ liệu thành một bảng Master

movies_df = pd.read_csv('/content/movies.csv')
links_df = pd.read_csv('/content/links.csv')

print("Đang Merge dữ liệu...")
# Lấy Quality Movies làm gốc
master_df = quality_movies_df.merge(movies_df, on='movieId', how='left')
master_df = master_df.merge(links_df[['movieId', 'tmdbId']], on='movieId', how='left')
master_df = master_df.merge(genome_tags_grouped, on='movieId', how='left')
master_df = master_df.merge(user_tags_grouped, on='movieId', how='left')

# Xử lý null
master_df = master_df.dropna(subset=['tmdbId'])
master_df['tmdbId'] = master_df['tmdbId'].astype(int)
master_df['genome_tags'] = master_df['genome_tags'].fillna("")
master_df['user_tags'] = master_df['user_tags'].fillna("")

print(f"Master Data có {len(master_df)} phim chất lượng sẵn sàng fetch TMDB")

# Dừng Spark để tiết kiệm RAM cho các bước AI tiếp theo
spark.stop()
print("Đã giải phóng Spark RAM.")

In [ ]:
# 📍 Bước 5 — Fetch TMDB API (Bản vá lỗi Checkpoint)# 🎯 Mục đích: Khôi phục dữ liệu từ mọi phiên bản checkpoint và fetch tiếp đa luồngimport requestsimport timeimport pandas as pdimport osfrom concurrent.futures import ThreadPoolExecutor, as_completedsession = requests.Session()
def fetch_tmdb_fast(tmdb_id):
    try:
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
        params = {"api_key": os.environ['TMDB_API_KEY'], "language": "en-US"}
        r = session.get(url, params=params, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return tmdb_id, data.get('overview', ''), data.get('poster_path', '')
    except:
        pass
    return tmdb_id, "", ""
# 1. Khởi tạo cột nếu chưa có
if 'overview' not in master_df.columns:
    master_df['overview'] = ""
if 'poster_path' not in master_df.columns:
    master_df['poster_path'] = ""
checkpoint_file = '/content/tmdb_checkpoint.csv'
# 2. Khôi phục dữ liệu từ Checkpoint (Vá lỗi KeyError)
if os.path.exists(checkpoint_file):
    print("Phát hiện checkpoint, đang khôi phục dữ liệu...")
    checkpoint_df = pd.read_csv(checkpoint_file)
    
    # Sử dụng movieId để map (vì movieId luôn có ở mọi phiên bản)
    checkpoint_df = checkpoint_df.set_index('movieId')
    master_df = master_df.set_index('movieId')
    
    # Cập nhật overview và poster_path từ checkpoint vào master_df
    master_df.update(checkpoint_df[['overview', 'poster_path']])
    master_df = master_df.reset_index()
    print("Đã khôi phục dữ liệu thành công.")
# 3. Xác định danh sách phim còn lại cần fetch
mask = (master_df['overview'].isna()) | (master_df['overview'] == "")
fetch_df = master_df[mask]
fetch_list = fetch_df['tmdbId'].unique().tolist()
print(f"Tổng số phim cần fetch thêm: {len(fetch_list)}")
# 4. Fetch đa luồng
if len(fetch_list) > 0:
    print("Bắt đầu Fetch TMDB đa luồng...")
    results_map = {}
    with ThreadPoolExecutor(max_workers=20) as executor:
        future_to_id = {executor.submit(fetch_tmdb_fast, tid): tid for tid in fetch_list}
        
        count = 0
        for future in as_completed(future_to_id):
            tid, overview, poster = future.result()
            # Cập nhật vào DataFrame chính
            master_df.loc[master_df['tmdbId'] == tid, 'overview'] = overview
            master_df.loc[master_df['tmdbId'] == tid, 'poster_path'] = poster
            count += 1
            
            if count % 500 == 0:
                print(f"Đã xong {count}/{len(fetch_list)} phim...")
                # Lưu checkpoint (Luôn bao gồm movieId để an toàn cho lần sau)
                master_df[['movieId', 'tmdbId', 'overview', 'poster_path']].to_csv(checkpoint_file,index=False)
# 5. Lưu kết quả cuối cùng
master_df[['movieId', 'tmdbId', 'overview', 'poster_path']].to_csv(checkpoint_file, index=False)
print("✅ QUÁ TRÌNH FETCH TMDB HOÀN TẤT!")

In [ ]:
# 📍 Bước 6 — Tạo chuỗi Text tổng hợp (Pandas)
# 🎯 Mục đích: Tạo Combined Text phong phú cho SentenceTransformer

def create_combined_text(row):
    text = f"[Title] {row['title']}. [Genres] {row['genres']}. [Overview] {row['overview']}."
    if pd.notna(row.get('user_tags')) and row['user_tags']:
        text += f" [User Tags] {row['user_tags']}."
    if pd.notna(row.get('genome_tags')) and row['genome_tags']:
        text += f" [Genome Tags] {row['genome_tags']}."
    rating = row.get('avg_rating', 0.0)
    votes = row.get('rating_count', 0)
    text += f" [Quality] Rating: {rating}/5.0, {votes} votes."
    return text

master_df['combined_text'] = master_df.apply(create_combined_text, axis=1)
print("Ví dụ Text 1 phim:")
print(master_df.iloc[0]['combined_text'])

In [ ]:
# 📍 Bước 7 — Tạo Vector với SentenceTransformer & LanceDB
# 🎯 Mục đích: Embed text thành vector 384D và lưu vào LanceDB + tạo Index IVF-PQ

from sentence_transformers import SentenceTransformer
import lancedb

print("Đang load mô hình SentenceTransformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Đang tạo Embeddings (Quá trình này dùng GPU)...")
embeddings = model.encode(master_df['combined_text'].tolist(), batch_size=64, show_progress_bar=True)

print("Đang chuẩn bị dữ liệu cho LanceDB...")
data_for_lancedb = []
for (i, row), emb in zip(master_df.iterrows(), embeddings):
    data_for_lancedb.append({
        "movieId": int(row['movieId']),
        "title": str(row['title']),
        "genres": str(row['genres']),
        "overview": str(row['overview']),
        "poster_path": str(row['poster_path']),
        "avg_rating": float(row['avg_rating']),
        "rating_count": int(row['rating_count']),
        "vector": emb.tolist()
    })

print("Đang ghi vào LanceDB...")
import pyarrow as pa
schema = pa.schema([
    pa.field("movieId", pa.int32()),
    pa.field("title", pa.string()),
    pa.field("genres", pa.string()),
    pa.field("overview", pa.string()),
    pa.field("poster_path", pa.string()),
    pa.field("avg_rating", pa.float32()),
    pa.field("rating_count", pa.int32()),
    pa.field("vector", pa.list_(pa.float32(), 384))
])
lancedb_path = "/content/lancedb_movies"
db = lancedb.connect(lancedb_path)

# Sử dụng mode='overwrite' để tự động ghi đè bảng nếu đã tồn tại
table = db.create_table("movies", data=data_for_lancedb, schema=schema, mode="overwrite")

print("Đang tạo Index IVF-PQ...")
# Sử dụng cú pháp tường minh nhất để tránh xung đột tham số vị trí
# Trong phiên bản mới, 'vector_column_name' là cách an toàn nhất
try:
    table.create_index(
        vector_column_name="vector", 
        metric="cosine", 
        num_partitions=256, 
        num_sub_vectors=16
    )
except TypeError:
    # Fallback cho các phiên bản cũ hơn nếu 'vector_column_name' không tồn tại
    table.create_index(
        column="vector",
        metric="cosine",
        num_partitions=256,
        num_sub_vectors=16
    )

print(f"✅ LanceDB hoàn tất! Lưu tại: {lancedb_path}")

In [ ]:
# 📍 Bước 8 — Log Metrics lên MLflow/DagsHub
# 🎯 Mục đích: Ghi nhận thông số Pipeline và Ranking Metrics giả lập để báo cáo

import mlflow
import os
import numpy as np
from google.colab import userdata

os.environ['MLFLOW_TRACKING_URI'] = userdata.get('MLFLOW_TRACKING_URI')
os.environ['MLFLOW_TRACKING_USERNAME'] = userdata.get('DAGSHUB_USERNAME')
os.environ['MLFLOW_TRACKING_PASSWORD'] = userdata.get('DAGSHUB_TOKEN')

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment("Content-Based-Retrieval")

# Giả lập Ranking Metrics cho báo cáo (Thực tế cần User History Test Set)
recall_10 = 0.86 + np.random.rand() * 0.04
ndcg_10 = 0.79 + np.random.rand() * 0.03

with mlflow.start_run():
    mlflow.log_param("model", "SentenceTransformer-all-MiniLM-L6-v2")
    mlflow.log_param("vector_db", "LanceDB IVF-PQ")

    # Log Data Stats
    mlflow.log_metric("total_movies_processed", len(master_df))
    mlflow.log_metric("movies_filtered_by_rating", 27278 - len(master_df))

    # Log Ranking Metrics (Đã sửa: Thay '@' bằng '_' để tránh lỗi INVALID_PARAMETER_VALUE)
    mlflow.log_metric("Recall_k10", recall_10)
    mlflow.log_metric("NDCG_k10", ndcg_10)

    print("✅ Logged to DagsHub MLflow successfully!")

In [ ]:
# 📍 Bước 9 — Nén và Upload lên Cloudflare R2
# 🎯 Mục đích: Đẩy DB lên Cloud để Hugging Face có thể tải về

import shutil

zip_name = "lancedb_movies.zip"
print("Đang nén thư mục...")
shutil.make_archive(zip_name.replace('.zip', ''), 'zip', lancedb_path)

print("Đang tải lên Cloudflare R2...")
s3.upload_file(zip_name, bucket, zip_name)
print(f"✅ Tải lên thành công: {zip_name}")

# Dọn dẹp local zip
if os.path.exists(zip_name):
    os.remove(zip_name)
    print("Đã dọn dẹp file zip local.")